In [1]:
from globus_compute_sdk import Executor
import numpy
import matplotlib
from globus_compute_sdk.serialize import ComputeSerializer, CombinedCode
from globus_compute_sdk import Executor
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML, display

In [2]:
"""
OpenMM GPU MD Simulation — Argon gas box
Saves trajectory to trajectory.npz for separate visualization.
"""

def openmm_simulation(N=125, box_nm=3.0, spacing=None, n_frames=100, steps_each=500):
    import numpy as np
    from openmm import (
        System, NonbondedForce, LangevinMiddleIntegrator,
        Platform, Vec3
    )
    from openmm.app import Topology, Simulation, Element
    from openmm.unit import (
        nanometer, femtoseconds, picosecond,
        kelvin, amu, kilojoules_per_mole
    )
    spacing = spacing or box_nm / (N ** (1/3))
    # ── Topology ──────────────────────────────────────────────────
    print("Building Argon system...")
    topology = Topology()
    chain    = topology.addChain()
    for _ in range(N):
        res = topology.addResidue("AR", chain)
        topology.addAtom("AR", Element.getBySymbol("Ar"), res)

    topology.setPeriodicBoxVectors((
        Vec3(box_nm, 0, 0) * nanometer,
        Vec3(0, box_nm, 0) * nanometer,
        Vec3(0, 0, box_nm) * nanometer,
        )
    )

    side = int(round(N ** (1/3)))
    positions = [
        Vec3(ix * spacing + 0.1, iy * spacing + 0.1, iz * spacing + 0.1) * nanometer
        for ix in range(side) for iy in range(side) for iz in range(side)
    ]

    # ── System / Force Field ──────────────────────────────────────
    system = System()
    system.setDefaultPeriodicBoxVectors(
        Vec3(box_nm, 0, 0) * nanometer,
        Vec3(0, box_nm, 0) * nanometer,
        Vec3(0, 0, box_nm) * nanometer,
    )

    nb = NonbondedForce()
    nb.setNonbondedMethod(NonbondedForce.CutoffPeriodic)
    nb.setCutoffDistance(1.2 * nanometer)

    for _ in range(N):
        system.addParticle(39.948 * amu)
        nb.addParticle(0.0, 0.3405 * nanometer, 0.9960 * kilojoules_per_mole)

    system.addForce(nb)

    # ── Platform (GPU → CPU fallback) ─────────────────────────────
    for name in ['CUDA', 'OpenCL', 'CPU']:
        try:
            platform = Platform.getPlatformByName(name)
            print(f"✓ Platform: {name}")
            break
        except Exception:
            continue

    # ── Simulation ────────────────────────────────────────────────
    integrator = LangevinMiddleIntegrator(300*kelvin, 1.0/picosecond, 2.0*femtoseconds)
    simulation = Simulation(topology, system, integrator, platform)
    simulation.context.setPositions(positions)

    print("Minimizing energy...")
    simulation.minimizeEnergy(maxIterations=500)

    print(f"Running {n_frames} frames × {steps_each} steps...")
    traj  = []
    temps = []

    for i in range(n_frames):
        simulation.step(steps_each)
        state = simulation.context.getState(getPositions=True, getEnergy=True)
        pos   = state.getPositions(asNumpy=True).value_in_unit(nanometer)
        ke    = state.getKineticEnergy().value_in_unit(kilojoules_per_mole)
        T_ins = 2 * ke / (3 * N * 8.314e-3)
        traj.append(pos.copy())
        temps.append(T_ins)
        if (i + 1) % 10 == 0:
            print(f"  Frame {i+1}/{n_frames}  T≈{T_ins:.0f} K")

    # ── Save ──────────────────────────────────────────────────────
    np.savez("trajectory.npz",
             positions=np.array(traj),   # shape: (n_frames, N, 3)
             temps=np.array(temps),
             box_nm=box_nm)
    
    return {
        "positions": np.array(traj),   # (n_frames, N, 3)
        "temps": np.array(temps),
        "box_nm": box_nm,
    }

In [3]:
uep_conf = {'account': "AuroraGPT", 'queue': 'debug', 
            # Polaris MEP is not set to user `worker_init`, but oddly `config_key` to set env
            'config_key': 'source /home/yadunand/setup_openmm.sh; which python3'}

epid = '9a947ba5-f537-4681-acf3-cc66485aadec'

serializer = ComputeSerializer(
    strategy_code=CombinedCode()
)

def hello():
    import platform
    import sys
    return f"Hello from Polaris' MEP: {platform.uname().node} Python:{sys.executable}"

with Executor(endpoint_id=epid,
              user_endpoint_config=uep_conf,
              serializer=serializer) as gce:

    fut = gce.submit(hello)

    print(fut.result())

SyntaxError: unterminated string literal (detected at line 3) (1378007399.py, line 3)

In [ ]:
fut = None
with Executor(endpoint_id=epid,
              user_endpoint_config=uep_conf,
              serializer=serializer) as gce:

    fut = gce.submit(openmm_simulation)
    fut.result() # Block and wait for results
    print("Simulation complete!")

In [ ]:

def visualize(data: dict):
    traj    = data["positions"]   # (n_frames, N, 3)
    temps   = data["temps"]
    box_nm  = float(data["box_nm"])
    n_frames, N, _ = traj.shape
    print(f"  {n_frames} frames, {N} atoms, box={box_nm} nm")

    fig = plt.figure(figsize=(13, 5), facecolor='#0d1117')

    ax3d = fig.add_subplot(121, projection='3d', facecolor='#0d1117')
    ax3d.set_title('Argon Gas — OpenMM GPU MD', color='white', pad=10)
    ax3d.tick_params(colors='#888')
    for pane in [ax3d.xaxis.pane, ax3d.yaxis.pane, ax3d.zaxis.pane]:
        pane.set_edgecolor('#333')
        pane.fill = False

    colors = plt.cm.plasma(np.linspace(0.2, 0.9, N))
    sc = ax3d.scatter(
        traj[0][:, 0], traj[0][:, 1], traj[0][:, 2],
        s=60, c=colors, alpha=0.85, depthshade=True
    )
    ax3d.set_xlim(0, box_nm); ax3d.set_ylim(0, box_nm); ax3d.set_zlim(0, box_nm)
    ax3d.set_xlabel('x (nm)', color='#aaa')
    ax3d.set_ylabel('y (nm)', color='#aaa')
    ax3d.set_zlabel('z (nm)', color='#aaa')

    ax2 = fig.add_subplot(122, facecolor='#161b22')
    ax2.set_xlim(0, n_frames); ax2.set_ylim(0, 600)
    ax2.set_title('Instantaneous Temperature', color='white')
    ax2.set_xlabel('Frame', color='#aaa'); ax2.set_ylabel('T (K)', color='#aaa')
    ax2.tick_params(colors='#888')
    ax2.axhline(300, color='#ff6b6b', lw=1, ls='--', label='Target 300 K')
    ax2.legend(facecolor='#0d1117', labelcolor='white', edgecolor='#333')
    for sp in ax2.spines.values():
        sp.set_edgecolor('#333')

    line_t, = ax2.plot([], [], color='#58a6ff', lw=1.5)
    frame_label = ax2.text(0.02, 0.92, '', transform=ax2.transAxes,
                           color='white', fontsize=9)

    plt.tight_layout(pad=2)
    plt.close()

    def update(frame):
        pos = traj[frame]
        sc._offsets3d = (pos[:, 0], pos[:, 1], pos[:, 2])
        line_t.set_data(range(frame + 1), temps[:frame + 1])
        frame_label.set_text(f'Frame {frame+1}/{n_frames}  T≈{temps[frame]:.0f} K')
        return sc, line_t, frame_label

    ani = animation.FuncAnimation(fig, update, frames=n_frames, interval=60, blit=False)
    display(HTML(ani.to_jshtml()))

In [ ]:
visualize(fut.result())